In [1]:
"""
Q-G-CTGAN: Quality-Aware Cluster-Conditioned Oversampling
via Intra-Cluster Synthetic Sample Filtering

Notebook 02 (Final): Non-Generative Baseline Methods, Full 16-Dataset Run
with Feature Scaling

This notebook re-runs the full baseline evaluation (None, SMOTE, ADASYN,
G-SMOTE across RF, LightGBM, MLP, 5-fold CV) on all 16 datasets, with
StandardScaler added to the pipeline.

RATIONALE FOR ADDING SCALING (not present in the original manuscript's
protocol): an initial unscaled run revealed that MLP AUC collapsed to
~0.51 (near-random) on unsw_nb15 regardless of oversampler, while
RF/LightGBM (scale-invariant) performed normally (AUC 0.87-0.92) on the
same folds. unsw_nb15 mixes raw byte/rate/load network-traffic features
(values up to 10^6+) with one-hot encoded categorical features (0/1),
an extreme feature-scale disparity that MLP's gradient-based optimizer
cannot handle without normalization. To keep the comparison across
datasets and methods fair, StandardScaler (fit on the training fold
only, applied to both train and test) is now applied uniformly across
all 16 datasets and all four oversampling methods, inserted before
oversampling (SMOTE/ADASYN/G-SMOTE are themselves distance-based and
are also sensitive to feature scale).

This replaces both 02a_baselines_small.ipynb and 02b_baselines_large.ipynb;
their unscaled result files are retained for reference but superseded by
02_baseline_results_scaled.csv.
"""

import os
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn_extra.gsmote import GeometricSMOTE

warnings.filterwarnings("ignore")

# -- Paths --------------------------------------------------
DATASET_DIR = "./datasets"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# -- Reproducibility ------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Dataset directory : {DATASET_DIR}")
print(f"Results directory : {RESULTS_DIR}")
print(f"Random state      : {RANDOM_STATE}")


## 1. Dataset Registry (all 16, ordered small to large for early feedback)

DATASET_NAMES = [
    "ecoli", "pima_diabetes", "secom", "thyroid_sick", "ibm_attrition",
    "yeast_me2", "wine_quality", "churn", "abalone_19", "pageblocks",
    "satellite", "credit_default", "mammography",
    "protein_homo", "unsw_nb15", "fraud_detection",
]

datasets = {}
for name in DATASET_NAMES:
    path = os.path.join(DATASET_DIR, f"{name}.csv")
    df = pd.read_csv(path)
    datasets[name] = df
    minority = int(df["target"].sum())
    ir = round((len(df) - minority) / minority, 2)
    print(f"  {name:<20}  n={len(df):>7,}  features={df.shape[1]-1:>4}  IR={ir:>7.1f}")

print(f"\nTotal datasets loaded: {len(datasets)}")


## 2. Classifier Factory

def get_classifiers(random_state=RANDOM_STATE):
    return {
        "RF": RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        ),
        "LGBM": LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            random_state=random_state, verbosity=-1, n_jobs=-1,
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(128, 64), activation="relu",
            alpha=0.001, random_state=random_state, max_iter=500,
        ),
    }


## 3. Oversampler Factory

def get_oversamplers(random_state=RANDOM_STATE):
    return {
        "NoOverSampling": None,
        "SMOTE": SMOTE(random_state=random_state),
        "ADASYN": ADASYN(random_state=random_state),
        "G-SMOTE": GeometricSMOTE(random_state=random_state),
    }


## 4. Evaluation Loop (with StandardScaler)
# Pipeline per fold: split -> scale (fit on train only) -> oversample
# (on scaled train) -> train classifier -> evaluate on scaled test.
# Incremental save after each dataset completes.

N_FOLDS = 5

results = []
run_start = time.time()
out_path = os.path.join(RESULTS_DIR, "02_baseline_results_scaled.csv")

for ds_name, df in datasets.items():
    ds_start = time.time()

    X = df.drop(columns=["target"]).values.astype(np.float64)
    y = df["target"].values

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for sampler_name, sampler in get_oversamplers().items():
        combo_start = time.time()

        for clf_name, clf in get_classifiers().items():

            fold_aucs = []
            fold_times = []

            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                t0 = time.time()

                # Scale: fit on training fold only, transform both
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                if sampler is not None:
                    try:
                        X_train_res, y_train_res = sampler.fit_resample(X_train_scaled, y_train)
                    except Exception as e:
                        print(f"  [WARN] {ds_name}/{sampler_name} fold {fold_idx}: "
                              f"resample failed ({e}); using original training data")
                        X_train_res, y_train_res = X_train_scaled, y_train
                else:
                    X_train_res, y_train_res = X_train_scaled, y_train

                clf_fold = get_classifiers()[clf_name]
                clf_fold.fit(X_train_res, y_train_res)
                y_prob = clf_fold.predict_proba(X_test_scaled)[:, 1]

                auc = roc_auc_score(y_test, y_prob)
                elapsed = time.time() - t0

                fold_aucs.append(auc)
                fold_times.append(elapsed)

            results.append({
                "dataset": ds_name,
                "oversampler": sampler_name,
                "classifier": clf_name,
                "auc_mean": np.mean(fold_aucs),
                "auc_std": np.std(fold_aucs),
                "time_mean_sec": np.mean(fold_times),
                "time_total_sec": np.sum(fold_times),
            })

            print(f"  [{ds_name:<18}] {sampler_name:<15} + {clf_name:<5}  "
                  f"AUC={np.mean(fold_aucs):.4f}+-{np.std(fold_aucs):.4f}  "
                  f"time={np.sum(fold_times):.1f}s")

        combo_elapsed = time.time() - combo_start
        pd.DataFrame(results).to_csv(out_path, index=False)

    ds_elapsed = time.time() - ds_start
    print(f"  === {ds_name} complete in {ds_elapsed:.1f}s "
          f"(results saved to {out_path}) ===\n")

run_elapsed = time.time() - run_start

results_df = pd.DataFrame(results)
results_df.to_csv(out_path, index=False)

print(f"\nTotal wall-clock time: {run_elapsed:.1f}s ({run_elapsed/60:.1f} min)")
print(f"Saved {len(results_df)} result rows to {out_path}")
print(f"Expected rows: {len(DATASET_NAMES)} datasets x 4 oversamplers x 3 classifiers = {len(DATASET_NAMES)*4*3}")

Dataset directory : ./datasets
Results directory : ./results
Random state      : 42
  ecoli                 n=    336  features=   7  IR=    8.6
  pima_diabetes         n=    768  features=   8  IR=    1.9
  secom                 n=  1,567  features= 474  IR=   14.1
  thyroid_sick          n=  3,772  features=  52  IR=   15.3
  ibm_attrition         n=  1,470  features=  47  IR=    5.2
  yeast_me2             n=  1,484  features=   8  IR=   28.1
  wine_quality          n=  4,898  features=  11  IR=   25.8
  churn                 n=  5,000  features=  20  IR=    6.1
  abalone_19            n=  4,177  features=  10  IR=  129.5
  pageblocks            n=  5,473  features=  10  IR=  194.5
  satellite             n=  6,430  features=  36  IR=    9.3
  credit_default        n= 30,000  features=  23  IR=    3.5
  mammography           n= 11,183  features=   6  IR=   42.0
  protein_homo          n=145,751  features=  74  IR=  111.5
  unsw_nb15             n=175,341  features= 183  IR=   99.4
 

In [3]:
"""
02_results_summary.ipynb (or a new cell in 02): Tabular summary of the
scaled baseline results (16 datasets x 4 oversamplers x 3 classifiers),
formatted in the style of the original manuscript's Tables 3-5
(one table per classifier, datasets as rows, methods as columns).
"""

import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 10)

results = pd.read_csv("./results/02_baseline_results_scaled.csv", keep_default_na=False)

# Dataset display order: original 11 first (manuscript order), then the
# 5 datasets added during the major revision, grouped by IR tier within
# each block for readability.
DATASET_ORDER = [
    # -- Original 11 (manuscript order, Table 2) --
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "yeast_me2", "mammography", "abalone_19", "wine_quality", "ecoli",
    "pageblocks", "protein_homo",
    # -- Added in major revision --
    "satellite", "churn", "secom", "thyroid_sick", "unsw_nb15",
]

METHOD_ORDER = ["NoOverSampling", "SMOTE", "ADASYN", "G-SMOTE"]
METHOD_LABELS = {"NoOverSampling": "None", "SMOTE": "SMOTE",
                  "ADASYN": "ADASYN", "G-SMOTE": "G-SMOTE"}

for clf in ["RF", "LGBM", "MLP"]:
    sub = results[results["classifier"] == clf]
    pivot = sub.pivot_table(index="dataset", columns="oversampler",
                             values="auc_mean", aggfunc="mean")
    pivot = pivot.reindex(index=DATASET_ORDER, columns=METHOD_ORDER)
    pivot = pivot.rename(columns=METHOD_LABELS)
    pivot["Average"] = pivot.mean(axis=1)

    print("=" * 80)
    print(f"AUC results with {clf} classifier (post-revision, 16 datasets, scaled pipeline)")
    print("=" * 80)
    print(pivot.round(4).to_string())
    print()
    print(f"Column average (mean over all {len(DATASET_ORDER)} datasets):")
    print(pivot.mean(axis=0).round(4))
    print()

# Save all three tables to a single Excel file for easy inclusion in the
# revision response / supplementary materials.
with pd.ExcelWriter("./results/02_summary_tables.xlsx") as writer:
    for clf in ["RF", "LGBM", "MLP"]:
        sub = results[results["classifier"] == clf]
        pivot = sub.pivot_table(index="dataset", columns="oversampler",
                                 values="auc_mean", aggfunc="mean")
        pivot = pivot.reindex(index=DATASET_ORDER, columns=METHOD_ORDER)
        pivot = pivot.rename(columns=METHOD_LABELS)
        pivot["Average"] = pivot.mean(axis=1)
        pivot.round(4).to_excel(writer, sheet_name=clf)

print("Saved all three tables to ./results/02_summary_tables.xlsx")

AUC results with RF classifier (post-revision, 16 datasets, scaled pipeline)
oversampler        None   SMOTE  ADASYN  G-SMOTE  Average
dataset                                                  
credit_default   0.7791  0.7746  0.7710   0.7673   0.7730
fraud_detection  0.9801  0.9844  0.9816   0.9843   0.9826
pima_diabetes    0.8296  0.8225  0.8199   0.8271   0.8248
ibm_attrition    0.8026  0.8151  0.8139   0.8039   0.8089
yeast_me2        0.9343  0.9270  0.9262   0.9409   0.9321
mammography      0.9419  0.9463  0.9404   0.9309   0.9399
abalone_19       0.7697  0.7782  0.7848   0.7743   0.7768
wine_quality     0.8667  0.8563  0.8517   0.8654   0.8600
ecoli            0.9276  0.9368  0.9411   0.9501   0.9389
pageblocks       0.9994  0.9993  0.9993   0.9994   0.9994
protein_homo     0.9923  0.9926  0.9913   0.9732   0.9873
satellite        0.9533  0.9569  0.9499   0.9539   0.9535
churn            0.9181  0.9185  0.9135   0.9153   0.9164
secom            0.7265  0.7161  0.7222   0.6987   0.

In [4]:
"""
Single-combination re-run: fraud_detection / NoOverSampling / LGBM

Investigates the anomalously low AUC (0.8591) observed for this specific
combination in the full 02 run, which stood out against the same
dataset's other oversampler results under LGBM (SMOTE 0.9797, ADASYN
0.9785, G-SMOTE 0.9788) and against the same combination's AUC under
RF (0.9801) and MLP (0.9605). The original 5-fold CV (random_state=42)
is re-run identically first to confirm reproducibility, then re-run
with a different random_state to check whether the result is sensitive
to the specific fold split.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

RESULTS_DIR = "./results"
DATASET_DIR = "./datasets"
N_FOLDS = 5

df = pd.read_csv(f"{DATASET_DIR}/fraud_detection.csv")
X = df.drop(columns=["target"]).values.astype(np.float64)
y = df["target"].values

def run_lgbm_none(X, y, random_state, n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    fold_aucs = []
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            random_state=random_state, verbosity=-1, n_jobs=-1,
        )
        clf.fit(X_train_scaled, y_train)
        y_prob = clf.predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        fold_aucs.append(auc)
        print(f"    fold {fold_idx}: AUC={auc:.4f}")
    return fold_aucs

print("=== Re-run with original random_state=42 (reproducibility check) ===")
aucs_42 = run_lgbm_none(X, y, random_state=42)
print(f"  Mean AUC: {np.mean(aucs_42):.4f} +- {np.std(aucs_42):.4f}\n")

print("=== Re-run with different random_state=123 (fold-sensitivity check) ===")
aucs_123 = run_lgbm_none(X, y, random_state=123)
print(f"  Mean AUC: {np.mean(aucs_123):.4f} +- {np.std(aucs_123):.4f}\n")

print("=== Re-run with different random_state=7 (fold-sensitivity check) ===")
aucs_7 = run_lgbm_none(X, y, random_state=7)
print(f"  Mean AUC: {np.mean(aucs_7):.4f} +- {np.std(aucs_7):.4f}\n")

print("Summary:")
print(f"  Original (seed=42, from full run) : 0.8591 (reported)")
print(f"  Reproduced (seed=42, this re-run)  : {np.mean(aucs_42):.4f}")
print(f"  seed=123                           : {np.mean(aucs_123):.4f}")
print(f"  seed=7                             : {np.mean(aucs_7):.4f}")

=== Re-run with original random_state=42 (reproducibility check) ===
    fold 0: AUC=0.8625
    fold 1: AUC=0.8617
    fold 2: AUC=0.8183
    fold 3: AUC=0.8545
    fold 4: AUC=0.8984
  Mean AUC: 0.8591 +- 0.0255

=== Re-run with different random_state=123 (fold-sensitivity check) ===
    fold 0: AUC=0.9538
    fold 1: AUC=0.6893
    fold 2: AUC=0.9297
    fold 3: AUC=0.9081
    fold 4: AUC=0.8119
  Mean AUC: 0.8586 +- 0.0974

=== Re-run with different random_state=7 (fold-sensitivity check) ===
    fold 0: AUC=0.8776
    fold 1: AUC=0.9457
    fold 2: AUC=0.9251
    fold 3: AUC=0.9261
    fold 4: AUC=0.8189
  Mean AUC: 0.8987 +- 0.0458

Summary:
  Original (seed=42, from full run) : 0.8591 (reported)
  Reproduced (seed=42, this re-run)  : 0.8591
  seed=123                           : 0.8586
  seed=7                             : 0.8987


In [5]:
"""
Q-G-CTGAN: Quality-Aware Cluster-Conditioned Oversampling
via Intra-Cluster Synthetic Sample Filtering

Notebook 02 (v2): Non-Generative Baseline Methods, Extended Metrics

Re-runs the full baseline evaluation (None, SMOTE, ADASYN, G-SMOTE
across RF, LightGBM, MLP, 5-fold CV) on all 16 datasets, now recording
the full extended metric suite (AUC, PR-AUC, F1, Precision, Recall,
G-mean, Balanced Accuracy) introduced in Notebooks 03/04a/04b/04c, to
close a gap in Appendix A (F1-score results) where these four methods
were originally missing.

This supersedes 02_baseline_results_scaled.csv with
02_baseline_results_extended.csv.
"""

import os
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, balanced_accuracy_score, confusion_matrix,
)
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn_extra.gsmote import GeometricSMOTE
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATASET_DIR = "./datasets"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATASET_NAMES = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "yeast_me2", "mammography", "abalone_19", "wine_quality",
    "ecoli", "pageblocks", "protein_homo",
    "satellite", "churn", "secom", "thyroid_sick", "unsw_nb15",
]

print(f"Datasets: {len(DATASET_NAMES)}")


def get_classifier(name, random_state=RANDOM_STATE):
    if name == "RF":
        return RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        )
    elif name == "LGBM":
        return LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            random_state=random_state, verbosity=-1, n_jobs=-1,
        )
    elif name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), activation="relu",
            alpha=0.001, random_state=random_state, max_iter=500,
        )


def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sensitivity * specificity))


def evaluate(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    return {
        "AUC"         : round(roc_auc_score(y_test, y_prob), 4),
        "PR_AUC"      : round(average_precision_score(y_test, y_prob), 4),
        "F1"          : round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Precision"   : round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Recall"      : round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "G_mean"      : round(g_mean_score(y_test, y_pred), 4),
        "Balanced_Acc": round(balanced_accuracy_score(y_test, y_pred), 4),
    }


def get_oversamplers(random_state=RANDOM_STATE):
    return {
        "NoOverSampling": None,
        "SMOTE": SMOTE(random_state=random_state),
        "ADASYN": ADASYN(random_state=random_state),
        "G-SMOTE": GeometricSMOTE(random_state=random_state),
    }


N_FOLDS = 5

results = []
out_path = os.path.join(RESULTS_DIR, "02_baseline_results_extended.csv")

for ds_name in DATASET_NAMES:
    path = os.path.join(DATASET_DIR, f"{ds_name}.csv")
    df = pd.read_csv(path)

    bool_cols = df.select_dtypes(include="bool").columns
    if len(bool_cols):
        df[bool_cols] = df[bool_cols].astype("float64")

    X = df.drop(columns=["target"]).values.astype(float)
    y = df["target"].values.astype(int)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    print(f"\n{'='*65}\nDataset: {ds_name}  n={len(X):,}\n{'='*65}")

    for sampler_name, sampler in get_oversamplers().items():
        for clf_name in ["RF", "LGBM", "MLP"]:

            fold_metrics = {k: [] for k in
                             ["AUC", "PR_AUC", "F1", "Precision", "Recall", "G_mean", "Balanced_Acc"]}
            fold_times = []

            for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                t0 = time.time()
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                if sampler is not None:
                    try:
                        X_train_res, y_train_res = sampler.fit_resample(X_train_scaled, y_train)
                    except Exception as e:
                        print(f"  [WARN] {ds_name}/{sampler_name} fold {fold_idx}: resample failed ({e})")
                        X_train_res, y_train_res = X_train_scaled, y_train
                else:
                    X_train_res, y_train_res = X_train_scaled, y_train

                clf = get_classifier(clf_name)
                clf.fit(X_train_res, y_train_res)
                metrics = evaluate(clf, X_test_scaled, y_test)
                elapsed = time.time() - t0

                for k, v in metrics.items():
                    fold_metrics[k].append(v)
                fold_times.append(elapsed)

            row = {
                "dataset": ds_name, "oversampler": sampler_name, "classifier": clf_name,
                "time_total_sec": round(np.sum(fold_times), 2),
            }
            for k in fold_metrics:
                row[k] = round(np.mean(fold_metrics[k]), 4)
            results.append(row)

            print(f"  [{ds_name:<18}] {sampler_name:<15} + {clf_name:<5}  "
                  f"AUC={row['AUC']:.4f}  F1={row['F1']:.4f}  [{row['time_total_sec']:.1f}s]")

    pd.DataFrame(results).to_csv(out_path, index=False)

print(f"\nTotal rows: {len(results)}  (expected {len(DATASET_NAMES)*4*3})")
print(f"Saved to: {out_path}")

Datasets: 16

Dataset: credit_default  n=30,000
  [credit_default    ] NoOverSampling  + RF     AUC=0.7791  F1=0.7023  [3.3s]
  [credit_default    ] NoOverSampling  + LGBM   AUC=0.7836  F1=0.6847  [0.7s]
  [credit_default    ] NoOverSampling  + MLP    AUC=0.6940  F1=0.6373  [159.5s]
  [credit_default    ] SMOTE           + RF     AUC=0.7746  F1=0.6984  [5.1s]
  [credit_default    ] SMOTE           + LGBM   AUC=0.7731  F1=0.6991  [1.1s]
  [credit_default    ] SMOTE           + MLP    AUC=0.6804  F1=0.6173  [243.1s]
  [credit_default    ] ADASYN          + RF     AUC=0.7710  F1=0.6853  [5.4s]
  [credit_default    ] ADASYN          + LGBM   AUC=0.7703  F1=0.6971  [1.5s]
  [credit_default    ] ADASYN          + MLP    AUC=0.6760  F1=0.6077  [220.0s]
  [credit_default    ] G-SMOTE         + RF     AUC=0.7673  F1=0.6847  [6.9s]
  [credit_default    ] G-SMOTE         + LGBM   AUC=0.7835  F1=0.6819  [3.1s]
  [credit_default    ] G-SMOTE         + MLP    AUC=0.6957  F1=0.6293  [245.0s]

Dataset

In [6]:
"""
Re-integrate Notebook 02 results using the extended-metrics version
(02_baseline_results_extended.csv), which now includes F1, PR-AUC,
G-mean, Balanced Accuracy for None/SMOTE/ADASYN/G-SMOTE, closing the
gap that left these four methods as N/A in Appendix A (F1-score) and
the extended-metrics summary.
"""

import os
import pandas as pd
import numpy as np

RESULTS_DIR = "./results"

df_02 = pd.read_csv(os.path.join(RESULTS_DIR, "02_baseline_results_extended.csv"), keep_default_na=False)
df_03 = pd.read_csv(os.path.join(RESULTS_DIR, "03_qgctgan_results.csv"), keep_default_na=False)
df_04a = pd.read_csv(os.path.join(RESULTS_DIR, "04a_new_baselines_results.csv"), keep_default_na=False)
df_04b = pd.read_csv(os.path.join(RESULTS_DIR, "04b_ctdgan_results.csv"), keep_default_na=False)
df_04c = pd.read_csv(os.path.join(RESULTS_DIR, "04c_ctabganplus_results.csv"), keep_default_na=False)

print(f"02 (extended): {len(df_02)} rows")
print(f"Columns: {list(df_02.columns)}")

# -- Normalize 02: now has full metric set, no longer NaN-filled --
df_02_norm = df_02.rename(columns={"oversampler": "method"})
df_02_norm["method"] = df_02_norm["method"].replace({"NoOverSampling": "None"})
df_02_norm = df_02_norm[["dataset", "method", "classifier", "AUC", "PR_AUC", "F1",
                          "Precision", "Recall", "G_mean", "Balanced_Acc"]].copy()
df_02_norm["generation_time"] = np.nan
df_02_norm["train_time"] = np.nan
df_02_norm["method_group"] = "non_generative"

# -- 03: unchanged, proposed configuration only --
df_03_main = df_03[df_03["oversampling"] == "Q-G-CTGAN_adaptive_MMDon"].copy()
df_03_main = df_03_main.rename(columns={"oversampling": "method"})
df_03_main["method"] = "Q-G-CTGAN"
df_03_norm = df_03_main[[
    "dataset", "method", "classifier", "AUC", "PR_AUC", "F1", "Precision",
    "Recall", "G_mean", "Balanced_Acc"
]].copy()
df_03_norm["generation_time"] = (
    df_03_main["gmm_time"] + df_03_main["ctgan_train_time"] +
    df_03_main["generation_time"] + df_03_main["filter_time"]
)
df_03_norm["train_time"] = df_03_main["train_time"]
df_03_norm["method_group"] = "proposed"

def normalize_04(df, method_group="new_generative"):
    out = df[["dataset", "method", "classifier", "AUC", "PR_AUC", "F1",
              "Precision", "Recall", "G_mean", "Balanced_Acc",
              "generation_time", "train_time"]].copy()
    out["method_group"] = method_group
    return out

df_04a_norm = normalize_04(df_04a)
df_04b_norm = normalize_04(df_04b)
df_04c_norm = normalize_04(df_04c)

unified = pd.concat([
    df_02_norm, df_03_norm, df_04a_norm, df_04b_norm, df_04c_norm
], ignore_index=True)

unified_path = os.path.join(RESULTS_DIR, "05_unified_results.csv")
unified.to_csv(unified_path, index=False)

print(f"\nUnified table: {len(unified)} rows")
print(f"Methods: {sorted(unified['method'].unique())}")

# Verify F1 is no longer N/A for the four previously-missing methods
check = unified[unified["method"].isin(["None", "SMOTE", "ADASYN", "G-SMOTE"])]
print(f"\nF1 non-null rate for None/SMOTE/ADASYN/G-SMOTE: "
      f"{check['F1'].notna().mean():.1%} (expected 100%)")

02 (extended): 192 rows
Columns: ['dataset', 'oversampler', 'classifier', 'time_total_sec', 'AUC', 'PR_AUC', 'F1', 'Precision', 'Recall', 'G_mean', 'Balanced_Acc']

Unified table: 420 rows
Methods: ['ADASYN', 'CTAB-GAN+', 'CTGAN_MOS', 'G-SMOTE', 'KMeans_CTGAN', 'None', 'Q-G-CTGAN', 'SMOTE', 'ctdGAN']

F1 non-null rate for None/SMOTE/ADASYN/G-SMOTE: 100.0% (expected 100%)


In [7]:
"""
Appendix A data: macro-averaged F1-score, RF classifier, all 16
datasets, all 9 methods -- now complete with the extended 02 results.
"""

DATASET_ORDER = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "churn", "ecoli", "satellite", "secom", "thyroid_sick", "wine_quality",
    "yeast_me2", "mammography", "abalone_19", "pageblocks",
    "protein_homo", "unsw_nb15",
]
DATASET_LABELS = {
    "credit_default": "Credit Default", "fraud_detection": "Fraud Detection",
    "pima_diabetes": "Pima Diabetes", "ibm_attrition": "IBM HR Attrition",
    "churn": "Churn", "ecoli": "Ecoli", "satellite": "Satellite",
    "secom": "SECOM", "thyroid_sick": "Thyroid Sick", "wine_quality": "Wine Quality",
    "yeast_me2": "Yeast ME2", "mammography": "Mammography", "abalone_19": "Abalone 19",
    "pageblocks": "PageBlocks", "protein_homo": "Protein Homology", "unsw_nb15": "UNSW-NB15",
}
METHOD_ORDER = ["None", "SMOTE", "ADASYN", "G-SMOTE", "KMeans_CTGAN",
                "CTGAN_MOS", "ctdGAN", "CTAB-GAN+", "Q-G-CTGAN"]

def make_latex_table_metric(classifier, metric_col):
    rows_tex = []
    col_sums = {m: [] for m in METHOD_ORDER}

    for ds in DATASET_ORDER:
        sub = unified[(unified["classifier"] == classifier) & (unified["dataset"] == ds)]
        vals = {}
        for m in METHOD_ORDER:
            row = sub[sub["method"] == m]
            if len(row) and row[metric_col].iloc[0] != "":
                vals[m] = float(row[metric_col].iloc[0])
            else:
                vals[m] = None
            if vals[m] is not None:
                col_sums[m].append(vals[m])

        present = {m: v for m, v in vals.items() if v is not None}
        ranked = sorted(present.items(), key=lambda kv: -kv[1])
        best_m = ranked[0][0] if ranked else None
        second_m = ranked[1][0] if len(ranked) > 1 else None

        cells = []
        for m in METHOD_ORDER:
            v = vals[m]
            if v is None:
                cells.append("N/A")
            else:
                s = f"{v:.4f}"
                if m == best_m:
                    s = f"\\textbf{{{s}}}"
                elif m == second_m:
                    s = f"\\underline{{{s}}}"
                cells.append(s)
        rows_tex.append(f"{DATASET_LABELS[ds]:<18}& " + " & ".join(cells) + r" \\")

    avg_cells = []
    for m in METHOD_ORDER:
        if m == "CTAB-GAN+":
            avg_cells.append("---")
        else:
            avg_cells.append(f"{np.mean(col_sums[m]):.4f}" if col_sums[m] else "N/A")
    numeric_avgs = [float(x) if x not in ("---", "N/A") else -1 for x in avg_cells]
    best_avg_idx = int(np.argmax(numeric_avgs))
    avg_cells[best_avg_idx] = f"\\textbf{{{avg_cells[best_avg_idx]}}}"
    rows_tex.append(r"\midrule")
    rows_tex.append("\\textbf{Average}$^{\\dagger}$    & " + " & ".join(avg_cells) + r" \\")

    return "\n".join(rows_tex)

print("F1-score, RF classifier, all 9 methods:")
print(make_latex_table_metric("RF", "F1"))

F1-score, RF classifier, all 9 methods:
Credit Default    & \textbf{0.7023} & \underline{0.6984} & 0.6853 & 0.6847 & 0.6791 & 0.6784 & 0.6746 & 0.6758 & 0.6786 \\
Fraud Detection   & \textbf{0.9180} & 0.8155 & 0.5973 & 0.7224 & \underline{0.8992} & 0.8857 & 0.7341 & N/A & 0.8934 \\
Pima Diabetes     & \textbf{0.7475} & 0.7415 & 0.7434 & \underline{0.7454} & 0.7057 & 0.7057 & 0.7038 & 0.6952 & 0.7053 \\
IBM HR Attrition  & 0.6561 & \textbf{0.6991} & \underline{0.6837} & 0.5517 & 0.5545 & 0.5439 & 0.5189 & 0.5473 & 0.5345 \\
Churn             & \textbf{0.9037} & \underline{0.8880} & 0.8822 & 0.8857 & 0.8655 & 0.8502 & 0.8461 & 0.8674 & 0.8732 \\
Ecoli             & 0.7716 & 0.7661 & 0.7764 & 0.7590 & \underline{0.8671} & 0.8671 & 0.7967 & N/A & \textbf{0.8890} \\
Satellite         & 0.7876 & \textbf{0.7941} & 0.7569 & 0.7737 & 0.7814 & \underline{0.7924} & 0.7678 & 0.7835 & 0.7651 \\
SECOM             & 0.4828 & \textbf{0.5336} & \underline{0.5250} & 0.4826 & 0.4830 & 0.4830 & 0.4830 & 0

In [ ]:
"""
Resume 02_baseline_results_extended.csv from unsw_nb15 only.
The previous 15 datasets are already saved incrementally; this cell
picks up where the run stalled (SMOTE + MLP on unsw_nb15, which hung
for >8 hours) and appends only the remaining unsw_nb15 rows.
"""

import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, balanced_accuracy_score, confusion_matrix,
)
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn_extra.gsmote import GeometricSMOTE

RANDOM_STATE = 42
N_FOLDS = 5
DATASET_DIR = "./datasets"
RESULTS_DIR = "./results"
out_path = os.path.join(RESULTS_DIR, "02_baseline_results_extended.csv")

# Load already-completed results (15 datasets)
existing_results = pd.read_csv(out_path, keep_default_na=False)
results = existing_results.to_dict("records")
print(f"Loaded {len(results)} existing rows (expected 15*4*3=180)")
print(f"Datasets completed so far: {sorted(existing_results['dataset'].unique())}")

# --- redefine helper functions (in case kernel was restarted) ---
def get_classifier(name, random_state=RANDOM_STATE):
    if name == "RF":
        return RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        )
    elif name == "LGBM":
        return LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            random_state=random_state, verbosity=-1, n_jobs=-1,
        )
    elif name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), activation="relu",
            alpha=0.001, random_state=random_state, max_iter=500,
        )

def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sensitivity * specificity))

def evaluate(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    return {
        "AUC"         : round(roc_auc_score(y_test, y_prob), 4),
        "PR_AUC"      : round(average_precision_score(y_test, y_prob), 4),
        "F1"          : round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Precision"   : round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Recall"      : round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "G_mean"      : round(g_mean_score(y_test, y_pred), 4),
        "Balanced_Acc": round(balanced_accuracy_score(y_test, y_pred), 4),
    }

def get_oversamplers(random_state=RANDOM_STATE):
    return {
        "NoOverSampling": None,
        "SMOTE": SMOTE(random_state=random_state),
        "ADASYN": ADASYN(random_state=random_state),
        "G-SMOTE": GeometricSMOTE(random_state=random_state),
    }

# --- resume unsw_nb15 only ---
ds_name = "unsw_nb15"
path = os.path.join(DATASET_DIR, f"{ds_name}.csv")
df = pd.read_csv(path)
bool_cols = df.select_dtypes(include="bool").columns
if len(bool_cols):
    df[bool_cols] = df[bool_cols].astype("float64")
X = df.drop(columns=["target"]).values.astype(float)
y = df["target"].values.astype(int)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"\n{'='*65}\nResuming: {ds_name}  n={len(X):,}\n{'='*65}")

# Skip combinations already fully completed for unsw_nb15 (if any partial rows exist)
already_done = set(
    (r["oversampler"], r["classifier"])
    for r in results if r["dataset"] == ds_name
)
print(f"Already completed for {ds_name}: {already_done}")

# MLP is capped with a per-fold timeout via early stopping to avoid a
# repeat of the >8h hang: enable sklearn's built-in validation-based
# early stopping so training halts once validation score plateaus,
# rather than running the full max_iter=500 regardless of convergence.
def get_classifier_safe(name, random_state=RANDOM_STATE):
    if name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), activation="relu",
            alpha=0.001, random_state=random_state, max_iter=500,
            early_stopping=True, n_iter_no_change=10, validation_fraction=0.1,
        )
    return get_classifier(name, random_state)

for sampler_name, sampler in get_oversamplers().items():
    for clf_name in ["RF", "LGBM", "MLP"]:
        if (sampler_name, clf_name) in already_done:
            print(f"  [SKIP] {sampler_name} + {clf_name} (already completed)")
            continue

        fold_metrics = {k: [] for k in
                         ["AUC", "PR_AUC", "F1", "Precision", "Recall", "G_mean", "Balanced_Acc"]}
        fold_times = []

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            t0 = time.time()
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            if sampler is not None:
                try:
                    X_train_res, y_train_res = sampler.fit_resample(X_train_scaled, y_train)
                except Exception as e:
                    print(f"  [WARN] fold {fold_idx}: resample failed ({e})")
                    X_train_res, y_train_res = X_train_scaled, y_train
            else:
                X_train_res, y_train_res = X_train_scaled, y_train

            clf = get_classifier_safe(clf_name)
            clf.fit(X_train_res, y_train_res)
            metrics = evaluate(clf, X_test_scaled, y_test)
            elapsed = time.time() - t0

            for k, v in metrics.items():
                fold_metrics[k].append(v)
            fold_times.append(elapsed)
            print(f"    fold {fold_idx}: AUC={metrics['AUC']:.4f}  [{elapsed:.1f}s]")

        row = {
            "dataset": ds_name, "oversampler": sampler_name, "classifier": clf_name,
            "time_total_sec": round(np.sum(fold_times), 2),
        }
        for k in fold_metrics:
            row[k] = round(np.mean(fold_metrics[k]), 4)
        results.append(row)

        pd.DataFrame(results).to_csv(out_path, index=False)
        print(f"  [{ds_name}] {sampler_name} + {clf_name}  AUC={row['AUC']:.4f}  "
              f"[{row['time_total_sec']:.1f}s] -- saved")

print(f"\nTotal rows: {len(results)}  (expected 16*4*3=192)")